In [ ]:
#Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Load data
train_df = pd.read_csv('train_sol.csv', index_col='ID')
test_df = pd.read_csv('test_sol.csv', index_col='ID')

train_df.head()

,Soil_Type,Region,Irrigation,pH,Moisture,Nitrogen,Phosphorus,Potassium,Organic_Matter,Electrical_Conductivity,Bulk_Density,Clay_Percent,Temperature,Rainfall_7d,Sunlight_hours,Slope,Suitability
ID,,,,,,,,,,,,,,,,,
1,Sandy,Hill,Sprinkler,6.501,25.144,45.528,33.742,122.187,NaN,0.472,1.697,6.190,19.980,23.595,6.439,6.823,Unfavorable
2,Clay,Plain,Sprinkler,6.322,22.391,119.376,62.453,218.362,2.464,1.786,1.350,41.372,25.102,5.063,10.392,1.038,Favorable
3,Sandy,Hill,Sprinkler,6.847,23.489,74.258,36.014,120.864,2.596,1.270,1.627,1.000,17.128,18.410,8.126,7.177,Favorable
4,Loam,Plain,Sprinkler,6.697,30.294,90.570,66.035,198.206,2.899,0.596,1.302,32.664,22.799,24.588,7.034,0.000,Favorable
5,Clay,Hill,NaN,6.740,24.471,81.934,58.793,142.101,4.334,1.289,1.402,53.710,15.147,7.216,4.151,14.352,Favorable


In [ ]:
#Subtasks 1-2(FE)
def subtask2(value):
    if value < 6.0:
        return 'Acid'
    elif value >= 6.0 and value <= 7.5:
        return 'Neutral'
    else:
        return 'Alkaline'

def apply_subtasks(df):
    df = df.copy()
    #Subtask 1
    df['Nutrient_Index'] = df['Nitrogen'] * 0.4 + df['Phosphorus'] * 0.3 + df['Potassium'] * 0.3
    df['Nutrient_Index'] = round(df['Nutrient_Index'], 5)
    #Subtask 2
    df['pH_Classfication'] = df['pH'].apply(subtask2)
    return df

train_df = apply_subtasks(train_df)
test_df = apply_subtasks(test_df)

sub1 = test_df['Nutrient_Index'].to_list()
sub2 = test_df['pH_Classfication'].to_list()

In [ ]:
#Subtask 3-4
def subtask3(train_df, test_df):
    train_median = train_df['Moisture'].median()
    #Should have used
    #train_median = np.nanmedian(train_df['Mositure'].values)
    sub3 = []
    for value in test_df['Moisture']:
        if pd.isnull(value) or value <= train_median:
            sub3.append(0)
        else:
            sub3.append(1)

    return sub3

def subtask4(train_df, test_df):
    sub4 = []
    for test_value in test_df['Soil_Type']:
        count = 0
        for train_value in train_df['Soil_Type']:
            if test_value == train_value:
                count += 1
        sub4.append(count)

    return sub4

sub3 = subtask3(train_df, test_df)
sub4 = subtask4(train_df, test_df)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

X = train_df.drop('Suitability', axis=1)
y = train_df['Suitability']

#Encode y
le = LabelEncoder()
y = le.fit_transform(y)

#Preprocessing
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)],
    verbose_feature_names_out=False,
    remainder='drop')

X_transformed = preprocessor.fit_transform(X)
test_transformed = preprocessor.transform(test_df)

#Put back in a df
features = preprocessor.get_feature_names_out()
X_transformed = pd.DataFrame(X_transformed, columns=features, index=X.index)
test_transformed = pd.DataFrame(test_transformed, columns=features, index=test_df.index)

#Poly features
poly = PolynomialFeatures(degree=1, include_bias=False)
X_transformed[num_cols] = poly.fit_transform(X_transformed[num_cols])
test_transformed[num_cols] = poly.transform(test_transformed[num_cols])

In [ ]:
#Choose a model
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

models = {
    'LR':LogisticRegression(random_state=42),
    'SVC_rbf':SVC(kernel='rbf'),
    'SVC_linear':SVC(kernel='linear'),
    'GNB':GaussianNB(),
    'BNB':BernoulliNB(),
    'RF':RandomForestClassifier(random_state=42)
}

def cv_eval(model):
    f1_scorer = make_scorer(f1_score, pos_label=1)
    cv = cross_val_score(model, X_transformed, y, cv=5, scoring=f1_scorer, n_jobs=-1)  #Unfavorable:1
    return cv.mean()

for name, model in models.items():
    print(f'{name} | f1: {cv_eval(model)}')


LR | f1: 0.81859572946903
SVC_rbf | f1: 0.81598290855407
SVC_linear | f1: 0.8195546753661459
GNB | f1: 0.7424312376871571
BNB | f1: 0.7201198846161406
RF | f1: 0.7793329317325831


In [ ]:
#Submission
model = LogisticRegression(random_state=42)
model.fit(X_transformed, y)

preds = model.predict(test_transformed)
preds = le.inverse_transform(preds)

datapointID = []
for index in test_df.index:
    datapointID.extend([index]*5)

answer = []
for i in range(len(test_df)):
    answer.extend([sub1[i], sub2[i], sub3[i], sub4[i], preds[i]])

output_df = pd.DataFrame({
    'subtaskID':[1,2,3,4,5] * len(test_df),
    'datapointID':datapointID,
    'answer':answer
})

output_df.head()

,subtaskID,datapointID,answer
0,1,2601,96.5909
1,2,2601,Neutral
2,3,2601,1
3,4,2601,365
4,5,2601,Favorable


In [ ]:
output_df.to_csv('submission.csv', index=False)